# 12章 自動化とデプロイ

## 12.1 コードのデプロイ

コードの**デプロイ**（__Deployment__）は、ソフトウェアを構築する上で一般的なプロセス。  
 → コードが提供する機能をエンドユーザーが利用できるようにすることを意味する。（開発環境→エンドユーザーが触れる本番環境という意？）

 - 新しいバージョンのAPIをサーバーに送り、他のコードがAPIを呼び出せるようにする
 - 後述の「クラウドへのデプロイ」にもあるように、APIをクラウドプロパイダーでホスティングする
 - マーケットプレイスでアプリの新バージョンを公開してユーザーが新しいバージョンをスマフォなどのデバイスにインストールできるようにする

ソフトウェアの開発段階では新しいアイデアを試し、絶えず変更を加えていく。一方、デプロイはコードが満足できる段階に達したことを意味する。新機能の追加やバグ修正のときに行う場合もあるが、いずれにせよコードが何らかの意味で「完成」し、その変更をエンドユーザーに使ってもらいたいときに行うものだ。

コードデプロイ環境は複数持つのが一般的で、例えば本番・テスト・ステージングといった環境がある。本番にデプロイする前に、まずテスト環境にデプロイすることもある。これは本番と同じコードを使うが、隔離された「サンドボックス」環境で、本番製品の機能には影響しない。つまり、製品の他の部分と相互作用する場所でコードをテストし、問題がないことを確認してから本番にデプロイする、という流れになる。

デプロイ関連の用語の1つに、**CI/CD**（__Continuos Integration__/__Continuous Deployment or Delivery__）がある。   
 → デプロイプロセスのビルドパイプライン全体を指し、テストの実行、セキュリティチェック、コンテナのビルドとデプロイなどが含まれる。

- **継続的インテグレーション**  
    開発がコードベースに変更を加え、それをVCSにコミットすると、CIサーバーはプロジェクトをビルドし、テストを実行、すべてがうまくいくことをチェックする。エラーがあれば、CIシステムがアラートを出し、開発者が修正できすようにする
- **継続的デリバリー**  
    CIパイプラインが実行されると、コードは自動でビルドされ、すべてのテストが実行される。テストにすべて合格すれば、そのコードは本番環境にデプロイできる状態になる。ただし、実際にデプロイする前には、人によるコードレビューや手動での承認といった最終ステップが必要である
- **継続的デプロイ**  
    すべてのテストにパスすれば、コードは自動的に本番環境にデプロイされる

CI/CDシステムの運用には、コードがバージョン管理されていること、優れたテストセットがあること、チーム全体が自動化に賛同していることが必要だ。手作業でのテストが不要になり、迅速なフィードバックが得られ、大規模なコードベースの開発を加速できる。ただしセットアップが複雑なことが多く、小規模なプロジェクトには向かない場合もある。こうしたシステムは別のDevOpsチームが管理することも多い。  
人気のあるツールとしては 
- Jenkins
- Travis CI
- Circle CI
- GitHub Actions

がある。

最近、CI/CDの技術が機械学習分野に普及してきている。CI/CDは、コードの変更によってトリガーされるだけでなく、モデルの学習データの変更やモデルのパフォーマンスの低下によってもトリガーされる。そうしたトリガーが発生すると、システムはモデルを再トレーニングし、再デプロイする。

## 12.2 自動化の例

### 12.2.1 コミット前フック

**コミット前フック**（__pre-commit hooks__）は、その名のとおりコミットの前に実行される。これはGitフックの一種で、Gitのアクション（コミットなど）の前後に実行されるカスタムスクリプトだ。VCSにコミットする前に問題を見つけられるので便利で、特にリンティングやフォーマッティングなど自動化しやすい作業に向く。…コミット前フック（pre-commit hooks）は、その名のとおりコミットの前に実行される。これはGitフックの一種で、Gitのアクション（コミットなど）の前後に実行されるカスタムスクリプトだ。VCSにコミットする前に問題を見つけられるので便利で、特にリンティングやフォーマッティングなど自動化しやすい作業に向く。リントの実行にフックを使えば、好ましくないコードを検出し、コードベースにコミットされるのを防げる。

pre-commit（https://pre-commit.com）は、**.yamlファイル**で構成を管理するフレームワークだ。**YAML**（__YAML Ain't Markup Language__）は構成ファイルによく使われる、人間が読みやすい形式のマークアップ言語である。pre-commitを使うには、Gitでコードの変更を追跡している必要がある。



```yaml
repos:
  - repo: https://github.com/psf/black-pre-commit-mirror # 1
    rev: 23.10.1 # 2
    hooks:
      - id: black
        language_version: python3.13.3 # 3
```

1: Blackのリポジトリを指定する
2: 別のバージョンのBlackに置き換えることも可能
3: 別のバージョンのPythonに置き換えることも可能（利用中のもの）

### 12.2.2 GitHub Actions

## 12.3 クラウドへのデプロイ
1. ローカルマシンにDockerコンテナを作成し、APIのコードと、そのコードが依存するライブラリを保存
2. このコンテナを、選択したクラウドプロファイラのシステム上のコンテナレジストリにアップロードする。コンテナレジストリには複数のコンテナを含めることが出来る
3. 選んだコンテナを実行するよう、クラウドプロパイダーに指示する。これによりコンテナ内のAPIコードがネットに公開され、ユーザーがアクセス出来るようになる。コンテナはじ、クラウドプロパイダのサーバー上で実行される


### 12.3.1 コンテナとDocker

### 12.3.2 Dockerコンテナの構築

```dockerfile
FROM python:3.10 # ①

COPY requiments.txt . # ②

RUN pip install --upgrade -r requiments.txt # ③

COPY . . # ④

CMD ["python", "-m", "uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"] # ⑤
```

1. FROMキーワードは、このイメージが既存の別のイメージに基づくことを意味する。この場合、新しいイメージはPython 3.10の公式イメージに基づく
2. 新しいイメージには、requirements.txtファイルのみを追加（COPY）する
3. この場合、pipを実行（RUN）してrequirements.txtで指定されたライブラリをインストールする
4. buildコマンドが実行されたフォルダ内のすべてのファイルをイメージにコピー（COPY）する。Python ライブラリをインストールした後にすべてのファイルをコピーすることで、残りのイメージの構築プロセスが実行される
5. CMDは、コンテナの実行時に実行されるコマンドを意味する。この場合、コマンドは「11.2.2 APIへの機能の追加」で説明したように、Uvicornを使ってAPIを起動する